<a href="https://colab.research.google.com/github/Danial-071/MM-RAG/blob/main/MM_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!wget -O swift-style-embeddings.pkl https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/95eJ0YJVtqTZhEd7RaUlew/processed-swift-style-with-embeddings.pkl


--2026-10-31 14:50:35--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/95eJ0YJVtqTZhEd7RaUlew/processed-swift-style-with-embeddings.pkl
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6285731 (6.0M) [application/octet-stream]
Saving to: ‘swift-style-embeddings.pkl’

swift-style-embeddi 100%[===================>]   5.99M  --.-KB/s    in 0.07s   

2026-10-31 14:50:36 (90.7 MB/s) - ‘swift-style-embeddings.pkl’ saved [6285731/6285731]



In [9]:
"""
Configuration settings for the Style Finder application.
"""




# Image processing settings
IMAGE_SIZE = (224, 224)
NORMALIZATION_MEAN = [0.485, 0.456, 0.406]
NORMALIZATION_STD = [0.229, 0.224, 0.225]

# Default similarity threshold
SIMILARITY_THRESHOLD = 0.8

# Number of alternatives to return from search
DEFAULT_ALTERNATIVES_COUNT = 5


In [2]:
# models/image_processor.py
"""
Module for image processing, encoding, and similarity matching.
This module handles the conversion of images to vectors and finding
similar images in the dataset.
"""

import torch
import torchvision.transforms as transforms
from torchvision.models import resnet50
from PIL import Image
import requests
import base64
from io import BytesIO
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class ImageProcessor:
    """
    Handles image processing, encoding, and similarity comparisons.
    """
    def __init__(self, image_size=(224, 224),
                 norm_mean=[0.485, 0.456, 0.406],
                 norm_std=[0.229, 0.224, 0.225]):
        """
        Initialize the image processor with a pre-trained ResNet50 model.

        Args:
            image_size (tuple): Target size for input images
            norm_mean (list): Normalization mean values for RGB channels
            norm_std (list): Normalization standard deviation values for RGB channels
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = resnet50(pretrained=True).to(self.device)
        self.model.eval()  # Set model to evaluation mode

        # Image preprocessing pipeline
        self.preprocess = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=norm_mean, std=norm_std),
        ])

    def encode_image(self, image_input, is_url=True):
        """
        Encode an image and extract its feature vector.

        Args:
            image_input: URL or local path to the image
            is_url: Whether the input is a URL (True) or a local file path (False)

        Returns:
            dict: Contains 'base64' string and 'vector' (feature embedding)
        """
        try:
            if is_url:
                # Fetch the image from URL
                response = requests.get(image_input)
                response.raise_for_status()
                image = Image.open(BytesIO(response.content)).convert("RGB")
            else:
                # Load the image from a local file
                image = Image.open(image_input).convert("RGB")

            # Convert image to Base64
            buffered = BytesIO()
            image.save(buffered, format="JPEG")
            base64_string = base64.b64encode(buffered.getvalue()).decode("utf-8")

            # Preprocess the image for ResNet50
            input_tensor = self.preprocess(image).unsqueeze(0).to(self.device)

            # Extract features using ResNet50
            with torch.no_grad():
                features = self.model(input_tensor)

            # Convert features to a NumPy array
            feature_vector = features.cpu().numpy().flatten()

            return {"base64": base64_string, "vector": feature_vector}
        except Exception as e:
            print(f"Error encoding image: {e}")
            return {"base64": None, "vector": None}


    def find_closest_match(self, user_vector, dataset):
        """
        Find the closest match in the dataset based on cosine similarity.

        Args:
            user_vector: Feature vector of the user-uploaded image
            dataset: DataFrame containing precomputed feature vectors

        Returns:
            tuple: (Closest matching row, similarity score)
        """
        try:
            dataset_vectors = np.vstack(dataset['Embedding'].dropna().values)
            similarities = cosine_similarity(user_vector.reshape(1, -1), dataset_vectors)

            # Find the index of the most similar vector
            closest_index = np.argmax(similarities)
            similarity_score = similarities[0][closest_index]

            # Retrieve the closest matching row
            closest_row = dataset.iloc[closest_index]
            return closest_row, similarity_score
        except Exception as e:
            print(f"Error finding closest match: {e}")
            return None, None


In [6]:
"""
Service for interacting with the Llama 3.2 Vision Instruct model.
"""

import logging
from openrouter import OpenRouter
import os
from google.colab import userdata

os.environ['OPENROUTER_API_KEY'] = userdata.get('API')

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class LlamaVisionService:
    """
    Provides methods to interact with the Llama 4 Instruct model.
    """

    def __init__():

      model = OpenRouter(model = 'nvidia/nemotron-3-ultra-550b-a55b:free')
      return model

    def generate_response(self, encoded_image, prompt):
        """
        Generate a response from the model based on an image and prompt.

        Args:
            encoded_image (str): Base64-encoded image string
            prompt (str): Text prompt to guide the model's response

        Returns:
            str: Model's response
        """
        try:
            logger.info("Sending request to LLM with prompt length: %d", len(prompt))

            # Create the messages object
            messages = [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": prompt
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": "data:image/jpeg;base64," + encoded_image,
                            }
                        }
                    ]
                }
            ]

            # Send the request to the model
            response = self.model.chat(messages=messages)

            # Extract and validate the response
            content = response['choices'][0]['message']['content']

            logger.info("Received response with length: %d", len(content))

            # Check if response appears to be truncated
            if len(content) >= 7900:  # Close to common model limits
                logger.warning("Response may be truncated (length: %d)", len(content))

            return content

        except Exception as e:
            logger.error("Error generating response: %s", str(e))
            return f"Error generating response: {e}"


    def generate_fashion_response(self, user_image_base64, matched_row, all_items,
                                 similarity_score, threshold=0.8):
        """
        Generate a fashion-specific response using role-based prompts.

        Args:
            user_image_base64: Base64-encoded user-uploaded image
            matched_row: The closest match row from the dataset
            all_items: DataFrame with all items related to the matched image
            similarity_score: Similarity score between user and matched images
            threshold: Minimum similarity for considering an exact match

        Returns:
            str: Detailed fashion response
        """
        # Generate a simpler list of items with prices and links
        items_list = []
        for _, row in all_items.iterrows():
            item_str = f"{row['Item Name']} (${row['Price']}): {row['Link']}"
            items_list.append(item_str)

        # Join items with clear separators
        items_description = "\n".join([f"- {item}" for item in items_list])

        if similarity_score >= threshold:
            # Simplified prompt focused on professional fashion analysis
            assistant_prompt = (
                f"You're conducting a professional retail catalog analysis. "
                f"This image shows standard clothing items available in department stores. "
                f"Focus exclusively on professional fashion analysis for a clothing retailer. "
                f"ITEM DETAILS (always include this section in your response):\n{items_description}\n\n"
                "Please:\n"
                "1. Identify and describe the clothing items objectively (colors, patterns, materials)\n"
                "2. Categorize the overall style (business, casual, etc.)\n"
                "3. Include the ITEM DETAILS section at the end\n\n"
                "This is for a professional retail catalog. Use formal, clinical language."
            )
        else:
            # Similar approach for non-exact matches
            assistant_prompt = (
                f"You're conducting a professional retail catalog analysis. "
                f"This image shows standard clothing items available in department stores. "
                f"Focus exclusively on professional fashion analysis for a clothing retailer. "
                f"SIMILAR ITEMS (always include this section in your response):\n{items_description}\n\n"
                "Please:\n"
                "1. Note these are similar but not exact items\n"
                "2. Identify clothing elements objectively (colors, patterns, materials)\n"
                "3. Include the SIMILAR ITEMS section at the end\n\n"
                "This is for a professional retail catalog. Use formal, clinical language."
            )

        # Send the prompt to the model
        response = self.generate_response(user_image_base64, assistant_prompt)

        # Check if response is incomplete
        if len(response) < 100:
            logger.info("Response appears incomplete, creating basic response")
            # Create a basic response with the item details
            section_header = "ITEM DETAILS:" if similarity_score >= threshold else "SIMILAR ITEMS:"
            response = f"# Fashion Analysis\n\nThis outfit features a collection of carefully coordinated pieces.\n\n{section_header}\n{items_description}"

        # Ensure the items list is included - this is crucial
        elif "ITEM DETAILS:" not in response and "SIMILAR ITEMS:" not in response:
            logger.info("Item details section missing from response")
            # Append to existing response
            section_header = "ITEM DETAILS:" if similarity_score >= threshold else "SIMILAR ITEMS:"
            response += f"\n\n{section_header}\n{items_description}"

        return response


In [7]:
"""
Utility functions for the Style Finder application.
"""

import logging
import re

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def get_all_items_for_image(image_url, dataset):
    """
    Get all items related to a specific image from the dataset.

    Args:
        image_url (str): The URL of the matched image
        dataset (DataFrame): Dataset containing outfit information

    Returns:
        DataFrame: All items related to the image
    """
    related_items = dataset[dataset['Image URL'] == image_url]
    logger.info(f"Found {len(related_items)} items related to image URL: {image_url}")
    return related_items


def format_alternatives_response(user_response, alternatives, similarity_score, threshold=0.8):
    """
    Append alternatives to the user response in a formatted way.

    Args:
        user_response (str): Original response from the model
        alternatives (dict): Dictionary of alternatives for each item
        similarity_score (float): Similarity score of the match
        threshold (float): Threshold for determining match quality

    Returns:
        str: Enhanced response with alternatives
    """
    # Check if user_response is problematic
    if not user_response or any(phrase in user_response for phrase in [
            "I'm not able to provide",
            "I cannot",
            "I apologize, but",
            "I don't feel comfortable"]):
        # Create a basic response if the model refused
        user_response = "## Fashion Analysis Results\n\nHere are the items detected in your image:"

    if similarity_score >= threshold:
        enhanced_response = user_response + "\n\n## Similar Items Found\n\nHere are some similar items we found:\n"
    else:
        enhanced_response = user_response + "\n\n## Similar Items Found\n\nHere are some visually similar items:\n"

    # Count items added to ensure we're not exceeding reasonable limits
    items_added = 0
    max_items = 10

    for item, alts in alternatives.items():
        enhanced_response += f"\n### {item}:\n"
        if alts:
            for alt in alts[:3]:  # Limit to 3 alternatives per item
                if items_added < max_items:
                    enhanced_response += f"- {alt['title']} for {alt['price']} from {alt['source']} ([Buy it here]({alt['link']}))\n"
                    items_added += 1
        else:
            enhanced_response += "- No alternatives found.\n"

    return enhanced_response


def process_response(response: str) -> str:
    """
    Process and escape problematic characters in the response.

    Args:
        response (str): The original response text

    Returns:
        str: Processed response with escaped characters and proper formatting
    """
    if not response:
        logger.warning("Empty response received")
        return "# Fashion Analysis\n\nNo detailed analysis was generated. Please refer to the item details below."

    # Check for rejection messages
    rejection_phrases = [
        "I'm not able to provide",
        "I cannot provide",
        "I apologize, but I cannot",
        "I don't feel comfortable",
        "violated our content policy"
    ]

    # If the model rejected but we still have item details, extract and format them
    if any(phrase in response for phrase in rejection_phrases):
        logger.warning("Model rejected the request, extracting item details")

        # Try to extract the item details section
        items_section = None

        if "ITEM DETAILS:" in response:
            # Extract everything after ITEM DETAILS:
            items_section = "## Item Details\n\n" + response.split("ITEM DETAILS:")[1].strip()
        elif "SIMILAR ITEMS:" in response:
            # Extract everything after SIMILAR ITEMS:
            items_section = "## Similar Items\n\n" + response.split("SIMILAR ITEMS:")[1].strip()

        if items_section:
            # Format item details with proper Markdown
            formatted_items = re.sub(r'^\* ', '- ', items_section, flags=re.MULTILINE)
            return "# Fashion Analysis\n\nHere are the items detected in your image:\n\n" + formatted_items
        else:
            # Return whatever we got with minimal processing
            return response.replace("$", "\\$")

    # Escape $ signs for Markdown
    processed = response.replace("$", "\\$")

    # Ensure important sections are properly formatted
    if "ITEM DETAILS:" in processed:
        processed = processed.replace("ITEM DETAILS:", "## Item Details")

    if "SIMILAR ITEMS:" in processed:
        processed = processed.replace("SIMILAR ITEMS:", "## Similar Items")

    # Add proper formatting to ensure aesthetic display
    if not processed.startswith("#"):
        processed = "# Fashion Analysis\n\n" + processed

    # Ensure all bullet points use consistent Markdown
    processed = re.sub(r'^\* ', '- ', processed, flags=re.MULTILINE)

    return processed

In [14]:
"""
Main application file for the Style Finder Gradio interface.
"""

import gradio as gr
import pandas as pd
import os
from tempfile import NamedTemporaryFile

# Import local modules


class StyleFinderApp:
    """
    Main application class that orchestrates the Style Finder workflow.
    """

    def __init__(self, dataset_path):
        """
        Initialize the Style Finder application.

        Args:
            dataset_path (str): Path to the dataset file

        Raises:
            FileNotFoundError: If the dataset file is not found
            ValueError: If the dataset is empty or invalid
        """
        # Load the dataset
        if not os.path.exists(dataset_path):
            raise FileNotFoundError(f"Dataset file not found: {dataset_path}")

        self.data = pd.read_pickle(dataset_path)
        if self.data.empty:
            raise ValueError("The loaded dataset is empty")

        # Initialize components
        self.image_processor = ImageProcessor(
            image_size=IMAGE_SIZE,
            norm_mean=NORMALIZATION_MEAN,
            norm_std=NORMALIZATION_STD
        )




    def process_image(self, image):
        """
        Process a user-uploaded image and generate a fashion response.

        Args:
            image: PIL image uploaded through Gradio

        Returns:
            str: Formatted response with fashion analysis
        """
        # Save the image temporarily if it's not already a file path
        if not isinstance(image, str):
            temp_file = NamedTemporaryFile(delete=False, suffix=".jpg")
            image_path = temp_file.name
            image.save(image_path)
        else:
            image_path = image

        # Step 1: Encode the image
        user_encoding = self.image_processor.encode_image(image_path, is_url=False)
        if user_encoding['vector'] is None:
            return "Error: Unable to process the image. Please try another image."

        # Step 2: Find the closest match
        closest_row, similarity_score = self.image_processor.find_closest_match(user_encoding['vector'], self.data)
        if closest_row is None:
            return "Error: Unable to find a match. Please try another image."

        print(f"Closest match: {closest_row['Item Name']} with similarity score {similarity_score:.2f}")

        # Step 3: Get all related items
        all_items = get_all_items_for_image(closest_row['Image URL'], self.data)
        if all_items.empty:
            return "Error: No items found for the matched image."

        # Step 4: Generate fashion response
        bot_response = self.llm_service.generate_fashion_response(
            user_image_base64=user_encoding['base64'],
            matched_row=closest_row,
            all_items=all_items,
            similarity_score=similarity_score,
            threshold = SIMILARITY_THRESHOLD
        )

        # Clean up temporary file
        if not isinstance(image, str):
            try:
                os.unlink(image_path)
            except:
                pass

        return process_response(bot_response)



def create_gradio_interface(app):
    """
    Create and configure the Gradio interface.

    Args:
        app (StyleFinderApp): Instance of the StyleFinderApp

    Returns:
        gr.Blocks: Configured Gradio interface
    """
    with gr.Blocks(theme=gr.themes.Soft(), title="Fashion Style Analyzer") as demo:
        # Introduction
        gr.Markdown(
            """
            # Fashion Style Analyzer

            Upload an image to analyze fashion elements and get detailed information about the items.
            This application combines computer vision, vector similarity, and large language models
            to provide detailed fashion analysis.
            """
        )

        # Example images section - moved higher up
        gr.Markdown("### Example Images")
        with gr.Row():
            # Display the images directly
            gr.Image(value="examples/test-1.png", label="Example 1", show_label=True, scale=1)
            gr.Image(value="examples/test-2.png", label="Example 2", show_label=True, scale=1)
            gr.Image(value="examples/test-3.png", label="Example 3", show_label=True, scale=1)

        # Example image buttons
        with gr.Row():
            example1_btn = gr.Button("Use Example 1")
            example2_btn = gr.Button("Use Example 2")
            example3_btn = gr.Button("Use Example 3")

        with gr.Row():
            with gr.Column(scale=1):
                # Image input component
                image_input = gr.Image(
                    type="pil",
                    label="Upload Fashion Image"
                )

                # Submit button
                submit_btn = gr.Button("Analyze Style", variant="primary")

                # Status indicator
                status = gr.Markdown("Ready to analyze.")

            with gr.Column(scale=2):
                # Output markdown component for displaying analysis results
                output = gr.Markdown(
                    label="Style Analysis Results",
                    height=700
                )

        # Event handlers
        # 1. Submit button click with processing indicator
        submit_btn.click(
            fn=lambda: "Analyzing image... This may take a few moments.",
            inputs=None,
            outputs=status
        ).then(
            fn=app.process_image,
            inputs=[image_input],
            outputs=output
        ).then(
            fn=lambda: "Analysis complete!",
            inputs=None,
            outputs=status
        )

        # 2. Example image buttons
        example1_btn.click(
            fn=lambda: "examples/test-1.png",
            inputs=None,
            outputs=image_input
        ).then(
            fn=lambda: "Example 1 loaded. Click 'Analyze Style' to process.",
            inputs=None,
            outputs=status
        )

        example2_btn.click(
            fn=lambda: "examples/test-2.png",
            inputs=None,
            outputs=image_input
        ).then(
            fn=lambda: "Example 2 loaded. Click 'Analyze Style' to process.",
            inputs=None,
            outputs=status
        )

        example3_btn.click(
            fn=lambda: "examples/test-3.png",
            inputs=None,
            outputs=image_input
        ).then(
            fn=lambda: "Example 3 loaded. Click 'Analyze Style' to process.",
            inputs=None,
            outputs=status
        )

        # Information about the application
        gr.Markdown(
            """
            ### About This Application

            This system analyzes fashion images using:

            - **Image Encoding**: Converting fashion images into numerical vectors
            - **Similarity Matching**: Finding visually similar items in a database
            - **Advanced AI**: Generating detailed descriptions of fashion elements

            The analyzer identifies garments, fabrics, colors, and styling details from images.
            The database includes information on outfits with brand and pricing details.
            """
        )

    return demo


if __name__ == "__main__":
    try:
        # Initialize the app with the dataset
        app = StyleFinderApp("swift-style-embeddings.pkl")

        # Create the Gradio interface
        demo = create_gradio_interface(app)

        # Launch the Gradio interface
        demo.launch(
            server_name="127.0.0.1",
            share=True,
            debug = True# Set to False if you don't want to create a public link

        )
    except Exception as e:
        print(f"Error starting the application: {str(e)}")

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:5000 <> https://48ddd4b737917a3216.gradio.live
Killing tunnel 127.0.0.1:7860 <> https://d8ac549bfee33ec2a8.gradio.live
